# Figure 1 — Building the system
> repair input structure → put it in a box → add water → minimize ("settle before you shake")

This is the first of the figure notebooks — see **`00_intro`** for scope and the two-tier setup. Here we take the raw PDB entry and turn it into something an MD engine can actually integrate: a repaired, solvated, energy-minimized system. **Run this notebook first** — it saves the prepared system (`system.xml` + the minimized coordinates) that `fig2_dynamics` and the others *load* instead of re-preparing, so they don't each redo the (stochastic) preparation.

In [ ]:
# --- environment on-ramp: make sure the MD stack + the module are importable in THIS kernel ---
import importlib.util, sys, os, subprocess
_missing = [m for m in ("openmm", "pdbfixer", "mdtraj", "py3Dmol") if importlib.util.find_spec(m) is None]
if _missing and "google.colab" in sys.modules:                          # Colab: provision the stack
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openmm", "pdbfixer", "mdtraj", "py3Dmol"], check=False)
    _missing = [m for m in _missing if importlib.util.find_spec(m) is None]
if _missing:                                                            # still missing -> almost always the WRONG KERNEL
    raise SystemExit(f"Missing in this kernel: {_missing}. Select your MD-tutorial conda kernel "
                     "(Kernel > Change Kernel). If it isn't built yet: conda env create -f environment.yml; "
                     "if it exists but is stale: conda env update -f environment.yml.")
_BASE = os.environ.get("MDTUTORIAL_BASE", "https://raw.githubusercontent.com/OWNER/REPO/main")
for _mod in ("mdtutorial.py", "mdtviz.py", "md_scalogram.py"):          # grab the shipped modules if absent
    if not os.path.exists(_mod) and importlib.util.find_spec(_mod[:-3]) is None:
        import urllib.request
        try: urllib.request.urlretrieve(f"{_BASE}/{_mod}", _mod); print("fetched", _mod)
        except Exception as e: print("Place", _mod, "next to this notebook.", e)
import mdtutorial as mdt, mdtviz
PYMOL = mdtviz.setup_pymol()                                            # find/provision headless PyMOL (panels skip if none)

In [ ]:
# --- configuration ---
import numpy as np, mdtraj as md
import matplotlib.pyplot as plt
OUT       = "trpcage_out"      # ONE shared output root -> fig1 saves the prepared system here, fig2+ load it
SEED      = 2024               # SOLVATION/prep seed (repair + water placement). Change it to build a different
                               #   water box; fig2+ then load whatever Figure 1 saved.

### 1.1  Fetch the structure and look at the NMR ensemble
An NMR structure is not a single set of coordinates — it is an **ensemble** of models, all consistent with the experimental restraints. That spread carries real information, but read it carefully: it reflects both genuine conformational heterogeneity *and* how tightly the data restrain each region (sparsely-restrained loops look floppy whether or not they truly are), so treat it as a *hint* at flexibility, not a direct measurement of it. We build the simulation from one model, but it is worth seeing the ensemble first. *(Drag to rotate; it cycles through the models on its own.)*

*Code: `fetch_pdb` (in `mdtutorial.py`) downloads the entry from the RCSB; `view_ensemble` (in `mdtviz.py`) animates the deposited models with **py3Dmol**.*

In [ ]:
pdb = mdt.fetch_pdb("1L2Y", OUT)
mdtviz.view_ensemble(pdb).show()

### 1.2  Repair — and why "nothing to fix" is still a real step
For a clean NMR structure PDBFixer finds no missing residues, atoms, or termini. But we do **not** trust the deposited hydrogens: they may not match our force field's residue templates (atom naming) or the protonation state at our chosen pH. So we strip them and rebuild: hydrogens are placed from standard geometric templates with protonation set by our chosen pH, then — because we hand the routine our force field — relaxed to it. That is where force-field consistency actually enters the pipeline; from here on every atom matches the residue templates the `System` will be built from.

*Code: `repair` (in `mdtutorial.py`) runs **PDBFixer** to fix missing atoms and drop heterogens, then calls OpenMM's `Modeller.addHydrogens` **with our force field**, so the new hydrogens are optimized to it rather than to the generic potential PDBFixer's default would use. That step minimizes on a Context, so it is seeded and pinned to the Reference platform for reproducibility (see `md_determinism_test`).*

In [ ]:
modeller = mdt.repair(pdb, SEED, OUT)     # strip H, rebuild with the FF (seeded, Reference platform)

### 1.2b  The same repair on a crystal structure — NMR is the *easy* case
Our system is an NMR ensemble, which is unusually clean (NMR resolves hydrogens, so the deposited models already carry them). **The great majority of deposited structures are X-ray crystallographic**, with a small but fast-growing minority from cryo-EM — and both typically need more repair than an NMR model: they rarely resolve hydrogens, so *all* of them are missing and must be built; there are crystallographic waters (and sometimes ions/ligands) to drop; and there are often alternate conformations (altlocs), disordered residues, or nonstandard residues. Running the identical pipeline on ubiquitin (1UBQ, X-ray 1.8 Å; Vijay-Kumar *et al.* 1987) shows the contrast.

And here is what "build the hydrogens" actually looks like — the **stick** view (a.k.a. licorice) draws every atom and bond explicitly, the representation to reach for when you care about individual atoms rather than overall shape: three residues of ubiquitin as deposited (heavy atoms only, left) vs. after repair (hydrogens in teal, right), Phe ring face-on.

*Code: `repair_report` (in `mdtutorial.py`) runs the same repair path and prints what it changed (missing residues/atoms, termini) for each structure; `hydrogen_sticks` (in `mdtviz.py`) renders the before/after stick views with headless open-source **PyMOL** (skips gracefully if PyMOL isn't installed).*

In [ ]:
ubq = mdt.fetch_pdb("1UBQ", OUT)
mdt.repair_report(mdt.outp("1L2Y.pdb", OUT), "NMR 1L2Y")
mdt.repair_report(ubq, "X-ray 1UBQ", save=mdt.outp("1UBQ_repaired.pdb", OUT))
import matplotlib.image as mpimg
_vd = os.path.join(OUT, "figures")
before, after = mdtviz.hydrogen_sticks(ubq, mdt.outp("1UBQ_repaired.pdb", OUT), _vd)
if after:
    figH, axH = plt.subplots(1, 2, figsize=(9, 4.5), facecolor="white")
    for a, f, t in zip(axH, (before, after), ("as deposited (X-ray): no hydrogens", "after repair: + hydrogens (teal)")):
        a.imshow(mpimg.imread(f)); a.axis("off"); a.set_title(t, fontsize=11)
    figH.suptitle("Repair up close — 1UBQ residues 4–6 (sticks)", fontsize=12); figH.tight_layout(); plt.show()

### 1.3  Box, water, counter-ions — then build the System and minimize
We wrap the peptide in a periodic box with ≈1 nm of padding, fill it with CHARMM-modified **TIP3P** water (Jorgensen *et al.* 1983), and add ions to neutralize the net charge (TC5b is +1 at pH 7, so one Cl⁻ goes in). The peptide goes from a few hundred atoms to a few thousand — almost all of it water. That ratio is the point: in explicit solvent you spend most of your compute on the solvent.

The `System` is where the force field becomes numbers. We use **CHARMM36m** (Huang *et al.* 2017) with **PME** for long-range electrostatics (Darden *et al.* 1993; smooth PME, Essmann *et al.* 1995), a 1 nm cutoff, and `HBonds` constraints so we can use a 2 fs timestep. Energy minimization removes the worst steric clashes from packing — a downhill walk on the potential-energy surface, not dynamics. We record the energy so we can *see* it settle. Then we **save the prepared system** for the other notebooks.

*Code: `solvate` (in `mdtutorial.py`) wraps OpenMM's `Modeller.addSolvent` (TIP3P box + neutralizing ions); `build_system` wraps `ForceField.createSystem` (PME, 1 nm cutoff, `HBonds`); `minimize` wraps `Simulation.minimizeEnergy` (a `LocalEnergyMinimizer`); `save_prepared` writes `system.xml` + the minimized PDB that fig2 loads.*

In [ ]:
modeller = mdt.solvate(modeller, SEED, OUT)
system, sim, platform, props = mdt.build_system(modeller, SEED)
mdt.print_hardware_report(mdt.hardware_report(sim.context))
min_positions, curve = mdt.minimize(sim, record_curve=True)
prep = mdt.PreparedSystem(sim.topology, system, platform, props, min_positions,
                          mdt.hardware_report(sim.context), modeller=modeller, energy_curve=curve)
mdt.save_prepared(prep, OUT)                              # PORT: fig2+ load this instead of re-prepping
print(f"prepared {system.getNumParticles()} atoms; PE {curve[0]:.0f} -> {curve[-1]:.0f} kJ/mol; saved under {OUT}/")

### Figure 1 — assemble the panels
Panels 1–3 are ray-traced headlessly by open-source PyMOL from the `stage*_*.pdb` snapshots. **Panel 1** shows the repaired fold as a **cartoon** (helix ribbons, loops, N→C blue→red — the representation everyone recognizes) with a translucent surface. **Panels 2–3** switch to the **all-atom** peptide (element-colored sticks under the same grey surface) and zoom out to frame the **entire periodic box**, so you can see just how small the solute is relative to the water that fills the cell. Panel 4 is the minimization curve from the live run. Everything is reproducible from the snapshots (they skip gracefully if PyMOL isn't available).

*Code: `cartoon_panels` (in `mdtviz.py`) ray-traces panels 1–3 with headless open-source **PyMOL** (skipped if it isn't installed); panel 4 replots the energy `curve` that `minimize` recorded back in §1.3.*

In [ ]:
panels = mdtviz.cartoon_panels(mdt.outp("stage2_repaired.pdb", OUT),
                               mdt.outp("stage3_solvated.pdb", OUT), os.path.join(OUT, "figures"))
titles = {"panel1_repair": "1. Repair\n(cartoon + surface)", "panel2_box": "2. Box\n(all-atom, full cell)",
          "panel3_solvated": "3. Add water\n(TIP3P + ions)"}
shown = [(panels[k], titles[k]) for k in ("panel1_repair", "panel2_box", "panel3_solvated") if k in panels]
ncol = len(shown) + 1
fig = plt.figure(figsize=(4.25 * ncol, 5), facecolor="white")
gs = fig.add_gridspec(1, ncol, wspace=0.04)
for i, (f, t) in enumerate(shown):
    ax = fig.add_subplot(gs[0, i]); ax.imshow(mdtviz.sqcrop(mpimg.imread(f))); ax.axis("off"); ax.set_title(t, fontsize=11)
ax = fig.add_subplot(gs[0, len(shown)])
ax.plot(np.array(curve) / 1000, "o-", ms=3, color="firebrick"); ax.set_box_aspect(1.0)
ax.set(xlabel="minimization step", title=f"{len(shown)+1}. Minimize")
ax.set_ylabel("PE (10³ kJ/mol)")                          # /1000 -> narrow labels; put the y-axis on the RIGHT so
ax.yaxis.set_label_position("right"); ax.yaxis.tick_right()   # panel 4's labels don't overlap panel 3 (panels 1-3 unchanged)
if not shown: print("(cartoon panels skipped — PyMOL unavailable; showing the minimization curve only)")
fig.suptitle("Figure 1 — Building the system:  repair → box → water → minimize", fontsize=13)
fig.savefig(mdt.outp("figure1.png", OUT), dpi=150, bbox_inches="tight"); plt.show()

### The finished solvated system (interactive)
Protein cartoon (N→C spectrum), faint water-oxygen spheres, green ion(s). Drag to rotate.

*Code: `view_solvated` (in `mdtviz.py`) is a small **py3Dmol** wrapper; waters are drawn as small, light oxygen spheres because py3Dmol's transparency is unreliable, so size and colour carry the faintness rather than true opacity.*

In [ ]:
mdtviz.view_solvated(mdt.outp("stage3_solvated.pdb", OUT)).show()